# 🎓 Taller Final de Estadística — Análisis de Datos
## Base de datos: Restaurantes USA v2

**Librerías utilizadas:** Pandas, NumPy, Matplotlib, Seaborn, Math

---

# 📦 Bloque 1: Reconocimiento de Datos
En este primer bloque importamos todas las librerías necesarias para el análisis y realizamos una exploración inicial de los datos para entender su estructura, tipos de variables y estadísticas básicas.

## 1.1 Importación de Librerías

In [ ]:
# 'pandas' es la librería principal para manipulación y análisis de datos.
# Nos permite trabajar con DataFrames (tablas) de forma eficiente.
# La importamos con el alias 'pd' por convención universal en ciencia de datos.
import pandas as pd

# 'numpy' es la librería fundamental para cálculos numéricos en Python.
# Nos provee funciones matemáticas y el tipo especial 'NaN' (Not a Number)
# que usaremos para representar valores faltantes. Alias: 'np'.
import numpy as np

# 'matplotlib.pyplot' es la librería base de visualización en Python.
# La usamos para crear y personalizar gráficos (barras, torta, histograma, etc.).
# Alias: 'plt'.
import matplotlib.pyplot as plt

# 'seaborn' es una librería de visualización estadística construida sobre matplotlib.
# Ofrece gráficos más atractivos y funciones especializadas como boxplots.
# Alias: 'sns'.
import seaborn as sns

# 'math' es la librería estándar de Python para funciones matemáticas básicas
# como raíz cuadrada, potencias, logaritmos, etc.
import math

# Configuramos el estilo visual de seaborn para que todos los gráficos
# se vean con un fondo limpio y profesional. 'whitegrid' añade una cuadrícula
# gris claro que facilita la lectura de valores.
sns.set_style('whitegrid')

# Configuramos matplotlib para que los gráficos se muestren directamente
# dentro del notebook (inline). Esto es necesario en Jupyter Notebook.
%matplotlib inline

print('✅ Todas las librerías fueron importadas correctamente.')

## 1.2 Lectura del archivo CSV

In [ ]:
# Usamos pd.read_csv() para leer el archivo CSV y cargarlo en un DataFrame.
# Un DataFrame es como una tabla de Excel dentro de Python: tiene filas, columnas,
# y cada columna puede tener un tipo de dato diferente (texto, número, fecha, etc.).
# NOTA: Asegúrate de que el archivo 'base_datos_restaurantes_USA_v2.csv' esté
# en la misma carpeta que este notebook.
df_original = pd.read_csv('base_datos_restaurantes_USA_v2.csv')

# Creamos una COPIA del DataFrame original. Esto es MUY IMPORTANTE porque
# durante la limpieza modificaremos los datos, y necesitamos conservar el
# DataFrame original intacto para poder comparar el 'antes' y 'después'
# (por ejemplo, en los boxplots comparativos de la sección de gráficos).
# .copy() crea una copia independiente; sin él, ambas variables apuntarían
# a los mismos datos en memoria y los cambios en uno afectarían al otro.
df = df_original.copy()

print(f'✅ Archivo CSV cargado exitosamente.')
print(f'   El dataset tiene {df.shape[0]} filas (registros) y {df.shape[1]} columnas (variables).')

## 1.3 Exploración inicial de los datos

In [ ]:
# .head() muestra las primeras 5 filas del DataFrame por defecto.
# Es la primera función que SIEMPRE debemos usar al cargar datos nuevos.
# Nos permite ver rápidamente: los nombres de las columnas, el tipo de datos
# que contienen, y si hay valores que parecen extraños a simple vista.
df.head()

In [ ]:
# .info() nos da un resumen técnico completo del DataFrame:
# - Cuántas filas y columnas tiene
# - El nombre de cada columna
# - Cuántos valores NO nulos tiene cada columna (los que sí tienen dato)
# - El tipo de dato de cada columna (int64=entero, float64=decimal, object=texto)
# - Cuánta memoria ocupa el DataFrame en RAM
# Esto es crucial para detectar columnas con datos faltantes (nulos).
df.info()

In [ ]:
# .describe() calcula estadísticas descriptivas automáticas SOLO para las
# columnas numéricas: count (cantidad de datos no nulos), mean (promedio),
# std (desviación estándar), min (mínimo), 25% (cuartil 1), 50% (mediana),
# 75% (cuartil 3), max (máximo).
# Es fundamental para detectar anomalías: por ejemplo, si la edad tiene
# un mínimo de -5 o un máximo de 300, algo está mal.
df.describe()

In [ ]:
# Verificamos cuántos valores nulos hay en CADA columna.
# .isnull() convierte cada celda en True (si es nulo) o False (si tiene valor).
# .sum() cuenta cuántos True hay por columna, dándonos el total de nulos.
# Esto nos dice exactamente dónde debemos enfocar la limpieza de datos.
print('🔍 CONTEO DE VALORES NULOS POR COLUMNA:')
df.isnull().sum()

---
# 🧹 Bloque 2: Limpieza de Datos
La limpieza de datos es el paso **MÁS IMPORTANTE** en cualquier análisis.
"Basura entra, basura sale": si los datos están sucios, nuestras conclusiones serán incorrectas.

En este bloque:
1. Eliminamos duplicados
2. Tratamos edades incoherentes
3. Rellenamos variables cualitativas nulas
4. Tratamos correos electrónicos faltantes

## 2.1 Eliminación de duplicados

In [ ]:
# Verificamos si existen registros duplicados basándonos en 'id_persona'.
# Cada persona debería tener un ID único. Si hay duplicados, significa que
# el mismo cliente aparece más de una vez, lo cual distorsionaría nuestros
# cálculos (conteos, promedios, etc.).
# .duplicated(subset='id_persona') marca como True las filas cuyo id_persona
# ya apareció antes. .sum() cuenta cuántos duplicados hay.
duplicados_antes = df.duplicated(subset='id_persona').sum()
print(f'🔄 Registros duplicados encontrados (por id_persona): {duplicados_antes}')

# .drop_duplicates() elimina las filas duplicadas.
# - subset='id_persona': solo busca duplicados en esta columna.
# - keep='first': conserva la PRIMERA aparición y elimina las siguientes.
# - inplace=True: modifica el DataFrame directamente en lugar de crear uno nuevo.
df.drop_duplicates(subset='id_persona', keep='first', inplace=True)

# Verificamos que los duplicados fueron eliminados correctamente.
duplicados_despues = df.duplicated(subset='id_persona').sum()
print(f'   Registros duplicados después de limpiar: {duplicados_despues}')
print(f'   ✅ Registros restantes: {df.shape[0]}')

## 2.2 Tratamiento de la variable 'edad'

In [ ]:
# PROBLEMA: La edad tiene valores incoherentes. En el .describe() vimos que
# el mínimo es -5 (nadie tiene edad negativa) y el máximo es 300 (nadie vive tanto).

# Paso 2.2.1: Creamos una columna temporal llamada 'rango_edad' para
# CATEGORIZAR cada edad en grupos y así visualizar mejor qué valores son válidos.
# pd.cut() divide un rango numérico continuo en intervalos (bins) discretos.
bins = [-float('inf'), 0, 18, 35, 50, 65, 100, float('inf')]
labels = ['Negativo (inválido)', 'Joven (0-18)', 'Adulto joven (19-35)',
          'Adulto (36-50)', 'Adulto mayor (51-65)', 'Tercera edad (66-100)',
          'Excesivo (inválido)']

df['rango_edad'] = pd.cut(df['edad'], bins=bins, labels=labels, right=True)

# Mostramos cuántas personas caen en cada categoría de edad.
print('📊 Distribución de edades por categoría:')
df['rango_edad'].value_counts(dropna=False)

In [ ]:
# Paso 2.2.2: Convertimos a NaN los valores de edad que son INCOHERENTES.
# Definimos como 'incoherente' toda edad que sea menor a 0 O mayor a 100.
# El operador | significa 'O' (OR lógico): basta con que se cumpla UNA condición.
# .loc[condición, 'columna'] = valor: modifica SOLO las filas que cumplen la condición.
# np.nan es el valor especial 'Not a Number' de NumPy, que Pandas entiende como 'vacío'.
edades_invalidas = ((df['edad'] < 0) | (df['edad'] > 100)).sum()
print(f'⚠️  Edades incoherentes detectadas (< 0 o > 100): {edades_invalidas}')

df.loc[(df['edad'] < 0) | (df['edad'] > 100), 'edad'] = np.nan

print(f'   Valores nulos en edad después de invalidar: {df["edad"].isnull().sum()}')

In [ ]:
# Paso 2.2.3: Rellenamos las edades nulas con el PROMEDIO de edad agrupado
# por la variable 'consume_licor'. ¿Por qué agrupamos por 'consume_licor'?
# Porque el consumo de licor está correlacionado con la edad (los menores de edad
# normalmente no consumen licor). Así, si una persona que SÍ consume licor tiene
# edad faltante, le asignamos el promedio de quienes SÍ consumen, y viceversa.

# Calculamos el promedio de edad POR GRUPO de 'consume_licor'.
promedio_edad_por_licor = df.groupby('consume_licor')['edad'].mean()
print('📈 Promedio de edad por grupo de consume_licor:')
print(promedio_edad_por_licor)

# .transform() aplica una función a cada grupo y devuelve un resultado del MISMO
# tamaño que el DataFrame original. .fillna() reemplaza los NaN.
# .round(0) redondea al entero más cercano (la edad debe ser un número entero).
df['edad'] = df['edad'].fillna(
    df.groupby('consume_licor')['edad'].transform('mean')
).round(0)

print(f'\n   Valores nulos en edad después de rellenar: {df["edad"].isnull().sum()}')
print('   ✅ Edad limpia y completa.')

## 2.3 Tratamiento de variables cualitativas

In [ ]:
# PROBLEMA: La columna 'preferencias_alimenticias' tiene valores nulos.
# Para variables cualitativas NO podemos usar el promedio.
# En su lugar, usamos la MODA: el valor más frecuente.
# Lo hacemos agrupado por 'ciudad_residencia', porque las preferencias
# alimenticias pueden variar según la ciudad.

print(f'🍽️  Nulos en preferencias_alimenticias antes: {df["preferencias_alimenticias"].isnull().sum()}')

# ESTRATEGIA: Calculamos la moda por ciudad y luego la mapeamos a cada fila.
# Paso 1: Calculamos la moda (valor más frecuente) por ciudad.
moda_por_ciudad = df.groupby('ciudad_residencia')['preferencias_alimenticias'].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

print('\n   Moda de preferencia alimenticia por ciudad:')
print(moda_por_ciudad)

# Paso 2: .map() busca el valor de moda correspondiente a la ciudad de cada fila.
valores_de_relleno = df['ciudad_residencia'].map(moda_por_ciudad)

# Paso 3: .fillna() reemplaza SOLO los NaN con el valor mapeado.
df['preferencias_alimenticias'] = df['preferencias_alimenticias'].fillna(valores_de_relleno)

print(f'\n   Nulos en preferencias_alimenticias después: {df["preferencias_alimenticias"].isnull().sum()}')
print('   ✅ Preferencias alimenticias completas.')

## 2.4 Tratamiento de correo electrónico y limpieza final

In [ ]:
# Los correos electrónicos faltantes se rellenan con el texto 'NA' (Not Available).
print(f'📧 Nulos en correo_electronico antes: {df["correo_electronico"].isnull().sum()}')

df['correo_electronico'] = df['correo_electronico'].fillna('NA')

print(f'   Nulos en correo_electronico después: {df["correo_electronico"].isnull().sum()}')
print('   ✅ Correos electrónicos completos.')

# Eliminamos la columna temporal 'rango_edad' que ya no necesitamos.
df.drop(columns=['rango_edad'], inplace=True)
print('\n🗑️  Columna temporal rango_edad eliminada.')

## 2.5 Guardado del DataFrame limpio

In [ ]:
# Exportamos el DataFrame limpio a un nuevo archivo CSV.
# index=False: NO incluye el índice numérico de Python como columna.
df.to_csv('clientes_final.csv', index=False)

print(f'💾 DataFrame limpio guardado como clientes_final.csv')
print(f'   Registros finales: {df.shape[0]} filas x {df.shape[1]} columnas')

# Verificación final de nulos restantes
print('\n🔍 VERIFICACIÓN FINAL - NULOS RESTANTES:')
df.isnull().sum()

---
# 📊 Bloque 3: Estadísticos de Comparación
En este bloque creamos funciones reutilizables para calcular estadísticas de comparación entre grupos.

In [ ]:
# Leemos la base de datos LIMPIA para asegurarnos de trabajar con datos correctos.
df_limpio = pd.read_csv('clientes_final.csv')

## 3.1 Función `calcular_promedio`

In [ ]:
def calcular_promedio(df, columna_grupo):
    """
    Calcula el promedio de las principales variables numéricas agrupadas
    por una columna categórica del DataFrame.

    Parámetros:
    -----------
    df : pandas.DataFrame
        El DataFrame con los datos a analizar.
    columna_grupo : str
        Nombre de la columna categórica por la cual se agruparán los datos.

    Retorna:
    --------
    pandas.DataFrame
        Tabla con promedios por grupo.
    """
    # Definimos las columnas numéricas de las cuales queremos el promedio.
    columnas_numericas = ['edad', 'frecuencia_visita',
                          'promedio_gasto_comida', 'ingresos_mensuales']

    # .groupby() agrupa, [columnas] selecciona, .mean() calcula promedio, .round(2) redondea.
    resultado = df.groupby(columna_grupo)[columnas_numericas].mean().round(2)

    return resultado

In [ ]:
# Promedios agrupados por ciudad de residencia
print('📈 Promedio de variables numéricas POR CIUDAD:')
calcular_promedio(df_limpio, 'ciudad_residencia')

In [ ]:
# Promedios agrupados por género
print('📈 Promedio de variables numéricas POR GÉNERO:')
calcular_promedio(df_limpio, 'genero')

## 3.2 Función `tabla_cruzada`

In [ ]:
def tabla_cruzada(df, col_filas, col_columnas):
    """
    Genera una tabla cruzada (de contingencia) entre dos variables categóricas,
    mostrando los PORCENTAJES por fila.

    Parámetros:
    -----------
    df : pandas.DataFrame
        El DataFrame con los datos a analizar.
    col_filas : str
        Columna que irá en las FILAS de la tabla.
    col_columnas : str
        Columna que irá en las COLUMNAS de la tabla.

    Retorna:
    --------
    pandas.DataFrame
        Tabla cruzada con porcentajes por fila (cada fila suma 100%).
    """
    # pd.crosstab() crea la tabla cruzada.
    # normalize='index': normaliza POR FILA (cada fila sumará 100%).
    # * 100: convertimos proporciones a porcentajes.
    tabla = pd.crosstab(df[col_filas], df[col_columnas], normalize='index') * 100
    tabla = tabla.round(2)

    return tabla

In [ ]:
# Tabla cruzada: Ciudad de Residencia vs Género
print('📊 Tabla Cruzada: Ciudad vs Género (% por fila):')
tabla_cruzada(df_limpio, 'ciudad_residencia', 'genero')

In [ ]:
# Tabla cruzada: Ciudad vs Preferencias Alimenticias
print('📊 Tabla Cruzada: Ciudad vs Preferencias Alimenticias (% por fila):')
tabla_cruzada(df_limpio, 'ciudad_residencia', 'preferencias_alimenticias')

---
# 📈 Bloque 4: Gráficos y Análisis Visual
La visualización de datos es esencial para comunicar hallazgos de forma clara e intuitiva.

## 4.1 Diagrama de Barras: Clientes por Ciudad

In [ ]:
# Este gráfico muestra cuántos clientes hay en cada ciudad.
# plt.figure() crea una nueva 'figura' (lienzo) con tamaño 12x6 pulgadas.
plt.figure(figsize=(12, 6))

# sns.countplot() crea un gráfico de barras que CUENTA automáticamente.
# - hue='ciudad_residencia': asigna color por ciudad.
# - palette='viridis': paleta de colores profesional.
# - order=...: ordena las barras de mayor a menor.
ax = sns.countplot(data=df_limpio, x='ciudad_residencia',
                   hue='ciudad_residencia', palette='viridis', legend=False,
                   order=df_limpio['ciudad_residencia'].value_counts().index)

plt.title('Cantidad de Clientes por Ciudad de Residencia',
          fontsize=14, fontweight='bold')
plt.xlabel('Ciudad de Residencia', fontsize=12)
plt.ylabel('Cantidad de Clientes', fontsize=12)

# Rotamos las etiquetas del eje X a 45° para evitar superposición.
plt.xticks(rotation=45, ha='right')

# Añadimos el valor numérico exacto encima de cada barra.
for barra in ax.patches:
    ax.text(barra.get_x() + barra.get_width() / 2.,
            barra.get_height() + 50,
            f'{int(barra.get_height())}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('grafico_barras_clientes_por_ciudad.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 Diagramas de Bigotes (Boxplots) Comparativos de Edad

In [ ]:
# Creamos DOS boxplots lado a lado para comparar la distribución de edad
# ANTES y DESPUÉS de la limpieza de datos.
# Los boxplots son EXCELENTES para detectar VALORES ATÍPICOS (outliers).

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6))

# Boxplot IZQUIERDO: Datos originales (ANTES de limpiar)
sns.boxplot(y=df_original['edad'], ax=axes[0], color='salmon',
            flierprops=dict(marker='o', markerfacecolor='red', markersize=4))
axes[0].set_title('Edad - ANTES de Limpiar\n(Datos Originales)',
                  fontsize=13, fontweight='bold')
axes[0].set_ylabel('Edad', fontsize=12)

# Boxplot DERECHO: Datos limpios (DESPUÉS de limpiar)
sns.boxplot(y=df_limpio['edad'], ax=axes[1], color='lightgreen',
            flierprops=dict(marker='o', markerfacecolor='green', markersize=4))
axes[1].set_title('Edad - DESPUÉS de Limpiar\n(Datos Corregidos)',
                  fontsize=13, fontweight='bold')
axes[1].set_ylabel('Edad', fontsize=12)

plt.suptitle('Comparación de la Distribución de Edad: Antes vs Después de la Limpieza',
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('grafico_boxplots_edad_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Diagrama de Torta (Pie Chart)

In [ ]:
# Diagrama de torta: Proporción de clientes por GÉNERO
# El pie chart muestra proporciones como 'tajadas' de un círculo.

conteo_genero = df_limpio['genero'].value_counts()

plt.figure(figsize=(8, 8))

colores_genero = ['#FF6B6B', '#4ECDC4']  # Rojo coral y turquesa
explode_genero = (0.03, 0.03)            # Ligera separación entre tajadas

# plt.pie() crea el diagrama de torta.
# - autopct='%1.1f%%': muestra el porcentaje dentro de cada tajada.
# - startangle=90: inicia la primera tajada a las 12 en punto.
# - shadow=True: añade una sombra sutil para efecto 3D.
plt.pie(conteo_genero.values,
        labels=conteo_genero.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colores_genero,
        explode=explode_genero,
        shadow=True,
        textprops={'fontsize': 13})

plt.title('Proporción de Clientes por Género',
          fontsize=14, fontweight='bold', pad=20)
plt.axis('equal')  # Asegura que el círculo sea perfecto

plt.tight_layout()
plt.savefig('grafico_torta_genero.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Diagrama de torta: Proporción por PREFERENCIA ALIMENTICIA
conteo_preferencias = df_limpio['preferencias_alimenticias'].value_counts()

plt.figure(figsize=(9, 9))

colores_pref = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
explode_pref = tuple([0.03] * len(conteo_preferencias))

plt.pie(conteo_preferencias.values,
        labels=conteo_preferencias.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=colores_pref,
        explode=explode_pref,
        shadow=True,
        textprops={'fontsize': 12})

plt.title('Proporción de Clientes por Preferencia Alimenticia',
          fontsize=14, fontweight='bold', pad=20)

plt.axis('equal')
plt.tight_layout()
plt.savefig('grafico_torta_preferencias.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Histogramas

In [ ]:
# Histograma: Distribución de INGRESOS MENSUALES
# Un histograma divide valores numéricos en intervalos (bins) y cuenta
# cuántas observaciones caen en cada intervalo.

plt.figure(figsize=(12, 6))

# sns.histplot() crea un histograma mejorado.
# - bins=30: divide el rango en 30 intervalos iguales.
# - kde=True: superpone una Curva de Densidad de Kernel.
sns.histplot(data=df_limpio, x='ingresos_mensuales', bins=30,
             kde=True, color='steelblue', edgecolor='white')

plt.title('Distribución de Ingresos Mensuales de los Clientes',
          fontsize=14, fontweight='bold')
plt.xlabel('Ingresos Mensuales (USD)', fontsize=12)
plt.ylabel('Frecuencia (Cantidad de Clientes)', fontsize=12)

# Líneas verticales para media y mediana.
media_ingresos = df_limpio['ingresos_mensuales'].mean()
mediana_ingresos = df_limpio['ingresos_mensuales'].median()

plt.axvline(x=media_ingresos, color='red', linestyle='--', linewidth=2,
            label=f'Media: ${media_ingresos:,.0f}')
plt.axvline(x=mediana_ingresos, color='orange', linestyle='-.', linewidth=2,
            label=f'Mediana: ${mediana_ingresos:,.0f}')

plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('grafico_histograma_ingresos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Histograma: Distribución del GASTO PROMEDIO EN COMIDA
plt.figure(figsize=(12, 6))

sns.histplot(data=df_limpio, x='promedio_gasto_comida', bins=30,
             kde=True, color='#2ecc71', edgecolor='white')

plt.title('Distribución del Gasto Promedio en Comida por Visita',
          fontsize=14, fontweight='bold')
plt.xlabel('Gasto Promedio en Comida (USD)', fontsize=12)
plt.ylabel('Frecuencia (Cantidad de Clientes)', fontsize=12)

media_gasto = df_limpio['promedio_gasto_comida'].mean()
mediana_gasto = df_limpio['promedio_gasto_comida'].median()

plt.axvline(x=media_gasto, color='red', linestyle='--', linewidth=2,
            label=f'Media: ${media_gasto:,.2f}')
plt.axvline(x=mediana_gasto, color='orange', linestyle='-.', linewidth=2,
            label=f'Mediana: ${mediana_gasto:,.2f}')

plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('grafico_histograma_gasto_comida.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 🏆 Resumen del Proceso Completado

| # | Archivo Generado | Descripción |
|---|---|---|
| 1 | `clientes_final.csv` | Base de datos limpia |
| 2 | `grafico_barras_clientes_por_ciudad.png` | Barras por ciudad |
| 3 | `grafico_boxplots_edad_comparativo.png` | Boxplots antes/después |
| 4 | `grafico_torta_genero.png` | Torta de género |
| 5 | `grafico_torta_preferencias.png` | Torta de preferencias |
| 6 | `grafico_histograma_ingresos.png` | Histograma de ingresos |
| 7 | `grafico_histograma_gasto_comida.png` | Histograma de gasto en comida |

**Funciones creadas:**
- `calcular_promedio(df, columna_grupo)` → Promedios por grupo
- `tabla_cruzada(df, col_filas, col_columnas)` → Tabla cruzada con %

✅ **¡Análisis completado exitosamente!**